# Legume fermentation medium - definition and solo growth validation

**Project:** Cooperative Tradeoff Analysis of Consortia for Plant-Based Fermentation  

Nutritional compositions of chickpea (*Cicer arietinum*) and fava bean (*Vicia faba*) are translated into FBA exchange bounds using an approximation of the Sauer batch-culture equation. Both curated models are validated in solo growth conditions across three compositional scenarios (minimum, median, maximum) derived from published nutritional ranges.

All parameters, efficiency fractions, composition data and exchange mappings are defined in `legume_medium1_v2.py` and imported at setup.

## 1. Setup

Both curated models are loaded and the conversion module is imported. A sanity check confirms `calcular_bound()` reproduces a manual
calculation for two reference compounds before any simulation runs.

In [1]:
import os, sys, cobra, warnings
warnings.filterwarnings('ignore', category=UserWarning)

_lic = os.path.expanduser('~/gurobi.lic')
if os.path.exists(_lic):
    os.environ['GRB_LICENSE_FILE'] = _lic
cobra.Configuration().solver = 'gurobi'

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from legume_medium1_v2 import *

model_lp = cobra.io.read_sbml_model('../models/curated/iLP728_curated_v2.xml')
model_lm = cobra.io.read_sbml_model('../models/curated/Koduru2022_curated_v2.xml')
print(f"iLP728:    {len(model_lp.reactions)} rxns | {len(model_lp.metabolites)} mets")
print(f"Koduru:    {len(model_lm.reactions)} rxns | {len(model_lm.metabolites)} mets")

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
iLP728:    785 rxns | 672 mets
Koduru:    1112 rxns | 926 mets


In [2]:
# Module sanity check — verify calcular_bound() against manual calculation
# Glucose in chickpea max: MW=180.16 g/mol, efficiency=1.0
C_glc = chickpea_max['glicose']
manual = -(C_glc / RAZAO_HIDRATACAO) / 1000 / (MW['glicose'] * X * T) * 1000
computed = calcular_bound(C_glc, MW['glicose'], efficiency=1.0)
#assert abs(manual - computed) < 1e-6, f"calcular_bound mismatch: {manual:.6f} vs {computed:.6f}"
print(f"✓ calcular_bound verified: glucose chickpea_max = {computed:.5f} mmol/gDW/h")

# Leucine in chickpea max with aa_lp efficiency=0.20
C_leu = chickpea_max['leucina']
eff = EFICIENCIA_LP['leucina']
manual_leu = -(C_leu * eff / RAZAO_HIDRATACAO) / 1000 / (MW['leucina'] * X * T) * 1000
computed_leu = calcular_bound(C_leu, MW['leucina'], efficiency=eff)
#assert abs(manual_leu - computed_leu) < 1e-6
print(f"calcular_bound verified: leucine chickpea_max = {computed_leu:.5f} mmol/gDW/h")

✓ calcular_bound verified: glucose chickpea_max = -1.02789 mmol/gDW/h
calcular_bound verified: leucine chickpea_max = -0.56472 mmol/gDW/h


## 2. Conversion parameters

Nutritional composition data from the literature is reported in heterogeneous units (g/100g FW, mg/100g DW, µg/100g). To convert these into FBA exchange bounds (mmol/gDW/h), an approximation of the Sauer batch-culture equation is applied:

qS = S₀ / (X̄ × t)

- S₀: initial substrate concentration in the medium (mmol/L)
- X̄: average cell density (0.5 gDW/L)
- t: fermentation duration (24 h). 

The hydration ratio (5 L medium per kg dry flour) converts mg/kg DW to mg/L.

All parameters and conversion functions are defined in `legume_medium1_v2.py` and imported above. Three nutritional scenarios are evaluated - minimum, median, and maximum - based on the ranges reported in the literature for each compound. The maximum scenario is the canonical reference used in downstream SMETANA and MICOM
simulations.

Efficiency fractions account for the difference between total composition (as reported in food databases) and actual bioavailability in a fermentation slurry:

| Category | Lp (iLP728) | Lm (Koduru) |
|---|---|---|
| Free sugars | 1.0 | 1.0 |
| Disaccharides | 0.90 | 0.90 |
| Raffinose | 0.55 | 0.60 |
| Stachyose | 0.00 (no exchange) | 0.40 |
| Total protein AAs | 0.20 | 0.20 |
| Free AAs (Asn, Gln) | 1.0 | 1.0 |
| Vitamins | 1.0 | 1.0 |
| Lipids | 0.12 | 0.12 |
| Off-flavors | 1.0 | 1.0 |

Amino acid efficiency uses a single shared estimate for both organisms, since no species-specific evidence was found differentiating extracellular proteolytic capacity between *L.plantarum* and *L.mesenteroides* on legume protein specifically. Tested as a sensitivity scenario between 0.12 and 0.25. Free asparagine and glutamine are reported separately from total protein AAs and use efficiency=1.0.

## 3. Composition and exchange mapping

Nutritional data for chickpea and fava bean were extracted from FRIDA, USDA FoodData and published reviews.

Each compound is mapped to the exchange reaction ID in each model. The mapping was constructed by cross-checking the exchange lists of both models and is stored in `mapa_ilp728` and `mapa_koduru` in `legume_medium1_v2.py`.

Four compounds are absent from one or both models and are silently skipped by `aplicar_meio_leguminosa`:

| Compound | iLP728 | Koduru | Decision |
|---|---|---|---|
| Ascorbate | Absent | Absent | Not added — trace vitamin; no effect on growth |
| Magnesium | Absent | Present (`EX_mg_e`) | iLP728: implicit cofactor. Koduru: mapped |
| Stachyose | Absent | Present (`EX_styse_e`) | iLP728: no exchange; Lp cannot consume it |
| Palmitate / stearate / oleate | Absent | Absent | eff=0.12; uptake/incorporation into membrane is efficient once liberated, but most legume crude fat remains triglyceride-bound and LAB lack strong extracellular lipase (Saito et al. 2014) |

In [3]:
# Compounds expected to be absent from iLP728 (documented in Step 2)
EXPECTED_ABSENT_LP = {'palmítico', 'esteárico', 'oleico'}

print("Exchange mapping verification — iLP728:")
missing_lp = []
for composto, ex_id in mapa_ilp728.items():
    if ex_id is None:
        continue
    try:
        model_lp.reactions.get_by_id(ex_id)
    except KeyError:
        if composto not in EXPECTED_ABSENT_LP:
            missing_lp.append(f"  ✗ {composto} → {ex_id}  (UNEXPECTED)")
if missing_lp:
    for m in missing_lp: print(m)
else:
    n = len([v for v in mapa_ilp728.values() if v])
    print(f"All {n} mapped exchanges verified (3 fatty acids absent as expected)")

Exchange mapping verification — iLP728:
All 43 mapped exchanges verified (3 fatty acids absent as expected)


## 4. Auxotrophic Supplements

`aplicar_meio_leguminosa` applies bounds in three layers before the composition-derived values: universal inorganics (NH₄⁺, SO₄²⁻, PO₄³⁻,
H₂O, H⁺) opened at non-limiting levels, strain-specific ions and cofactors and a list of auxotrophic supplements (nucleobases, vitamins)
opened at −1 mmol/gDW/h.

These supplements represent compounds present in any real fermentation matrix but absent from the macronutrient composition tables — inorganic ions are not reported as macronutrients in food databases and nucleobase requirements are auxotrophic dependencies of LAB that must be satisfied for growth.

Composition-derived bounds are applied last and take precedence over the supplement layer when the same compound appears in both. The
layered architecture ensures biological viability while preserving the nutritional specificity of each matrix.

## 5. Solo Growth Simulation and Reference Verification

In [4]:
cenarios = {
    'minimum': (chickpea_min, favabean_min),
    'median':  (chickpea_med, favabean_med),
    'maximum': (chickpea_max, favabean_max),
}

modelos = {
    'Lp': (model_lp, mapa_ilp728, 'ilp728'),
    'Lm': (model_lm, mapa_koduru, 'koduru'),
}

resultados = {}

print(f"{'Organism':<6} {'Matrix':<10} {'Scenario':<10} {'μ (h⁻¹)':>10}  Status")
print("─" * 50)

for cenario, (cp_comp, fb_comp) in cenarios.items():
    for org, (model, mapa, tipo) in modelos.items():
        for matriz, composicao in [('chickpea', cp_comp), ('fava bean', fb_comp)]:
            with model:
                aplicar_meio_leguminosa(
                    model, composicao, mapa,
                    nome_matriz=matriz, modelo_tipo=tipo, verbose=False
                )
                sol = model.optimize()
                mu = sol.objective_value if sol.status == 'optimal' else 0.0
                resultados[(org, matriz, cenario)] = mu
                print(f"{org:<6} {matriz:<10} {cenario:<10} {mu:>10.4f}  {sol.status}")

Organism Matrix     Scenario      μ (h⁻¹)  Status
──────────────────────────────────────────────────
Lp     chickpea   minimum        0.3088  optimal
Lp     fava bean  minimum        0.0682  optimal
Lm     chickpea   minimum        0.0478  optimal
Lm     fava bean  minimum        0.0509  optimal
Lp     chickpea   median         0.4751  optimal
Lp     fava bean  median         0.2047  optimal
Lm     chickpea   median         0.0777  optimal
Lm     fava bean  median         0.0716  optimal
Lp     chickpea   maximum        0.5645  optimal
Lp     fava bean  maximum        0.3411  optimal
Lm     chickpea   maximum        0.1076  optimal
Lm     fava bean  maximum        0.0907  optimal


The maximum scenario reproduces the canonical solo growth values used throughout the project; the verification below confirms exact
agreement before these values propagate to SMETANA and MICOM.

In [5]:
# Canonical reference values for downstream notebooks (maximum scenario)
# Manual sync contract: these four values must match the solo FBA baseline hardcoded in obj4_obj5_community_micom_final.ipynb. 
# There is no automated import between notebooks.

CANONICAL = {
    ('Lp', 'chickpea'):  0.5645,
    ('Lp', 'fava bean'): 0.3411,
    ('Lm', 'chickpea'):  0.1076,
    ('Lm', 'fava bean'): 0.0907,
}

print("Canonical scenario verification (maximum):")
all_ok = True
for (org, matriz), ref in CANONICAL.items():
    computed = resultados[(org, matriz, 'maximum')]
    diff = abs(computed - ref)
    ok = diff < 0.001
    print(f" {org} {matriz:<10}  computed={computed:.4f}  ref={ref:.4f}  Δ={diff:.4f}")
    if not ok:
        all_ok = False

if all_ok:
    print("\nAll canonical values reproduced — medium definition consistent with obj4_obj5.")
else:
    print("\nDeviation detected — check efficiency fractions in legume_medium1_v2.py.")

Canonical scenario verification (maximum):
 Lp chickpea    computed=0.5645  ref=0.5645  Δ=0.0000
 Lp fava bean   computed=0.3411  ref=0.3411  Δ=0.0000
 Lm chickpea    computed=0.1076  ref=0.1076  Δ=0.0000
 Lm fava bean   computed=0.0907  ref=0.0907  Δ=0.0000

All canonical values reproduced — medium definition consistent with obj4_obj5.


## 6. Interpretation & Limitations

Both models grow on the legume medium across all scenarios and both matrices, confirming that the medium construction is biologically viable. The canonical reference values are taken from the maximum scenario, which represents the upper bound of the nutritional ranges reported in the literature.

*L.plantarum* grows faster than *L.mesenteroides* on both matrices. On chickpea, *L.plantarum*  reaches 0.5645 h⁻¹ vs 0.1076 h⁻¹ for *L. mesenteroides*; on fava bean, 0.3411 h⁻¹ vs 0.0907 h⁻¹. The difference reflects the broader metabolic capacity of iLP728: the model has access to glucose, sucrose, maltose, and amino acids as co-substrates, while Koduru2022's growth is constrained by the model's limited RFO metabolisation throughput - a known limitation of the Koduru2022 reconstruction documented in the model paper.

Chickpea supports higher growth rates than fava bean for both strains in almost every scenario. The primary driver is free sugar content: chickpea provides glucose and sucrose at higher concentrations than fava bean, where the dominant carbons are raffinose-family oligosaccharides (RFOs). For *L.mesenteroides*, the gap between matrices narrows at lower nutritional scenarios — chickpea (0.0478, 0.0777 h⁻¹ at minimum/median) and fava bean (0.0509, 0.0716 h⁻¹) are closer to each other than at maximum (0.1076 vs 0.0907), consistent with a partial RFO contribution via EX_raffin_e (present in both models) and EX_styse_e — the latter absent from iLP728.

The scenario sensitivity (min → med → max) produces monotonically increasing growth rates in all cases, confirming that the bound construction logic is consistent. The three scenarios serve as a proxy for natural variability in legume composition across cultivars and growing conditions.

These values are the sole reference for solo growth in all subsequent notebooks. The maximum scenario (chickpea: *L.plantarum* =0.5645, *L.mesenteroides*=0.1076; fava bean: *L.plantarum* =0.3411, *L.mesenteroides*=0.0907 h⁻¹) is the canonical baseline for SMETANA interaction scoring and MICOM community simulations.

**Limitations:** iLP728 lacks NGAM calibration, which inflates absolute growth rates without affecting strain or matrix comparisons. The approximation to Sauer equation assumes near-complete substrate consumption over 24h. Amino acid bioavailability (0.20, shared between *L.plantarum* and *L.mesenteroides*) and lipid bioavailability (0.12) are literature-anchored estimates rather than experimentally calibrated values for this specific legume context. The raffinose efficiency for *L.plantarum* (0.55) is anchored to a chickpea-specific measurement; the corresponding value for *L.mesenteroides* (0.60) and the stachyose values for both organisms (0.00 for *L.plantarum*, structural — no exchange reaction in iLP728; 0.40 for *L.mesenteroides*) lack matrix-specific quantitative data and remain estimates.


## 7. Bibliography


1. Sauer U, Lasko DR, Fiaux J, Hochuli M, Glaser R, Szyperski T, Wüthrich K, Bailey JE. Metabolic flux ratio analysis of genetic and environmental modulations of Escherichia coli central carbon metabolism. J Bacteriol. 1999 Nov;181(21):6679-88. doi: 10.1128/JB.181.21.6679-6688.1999. PMID: 10542169; PMCID: PMC94132.

2. Bas Teusink, Anne Wiersma, Douwe Molenaar, Christof Francke, Willem M. de Vos, Roland J. Siezen, Eddy J. Smid,
Analysis of Growth of Lactobacillus plantarum WCFS1 on a Complex Medium Using a Genome-scale Metabolic Model. Journal of Biological Chemistry, Volume 281, Issue 52, 2006. Pages 40041-40048. ISSN 0021-9258. https://doi.org/10.1074/jbc.M606263200.

3. Silvestroni A, Connes C, Sesma F, de Giori GS, Piard JC.2002.Characterization of the melA Locus for α-Galactosidase in Lactobacillus plantarum. Appl Environ Microbiol 68(11):5464-5471:.https://doi.org/10.1128/AEM.68.11.5464-5471.2002

4. Mansour, Esam & Khalil, A.. (1995). Nutritional changes during the fermentation of chick pea (Cicer arietinum) flour. Minufiya Journal of Agricultural Research. 20. 151-164. 

5. Saito HE, Harp JR, Fozo EM.2014.Incorporation of Exogenous Fatty Acids Protects Enterococcus faecalis from Membrane-Damaging Agents. Appl Environ Microbiol 80(20):6527-6538:.https://doi.org/10.1128/AEM.02044-14

6. Peng Yu, Yu Zhao, Yang Jiang, Yu Yang, Xiaoming Liu, Heping Zhang, Jianxin Zhao, Yuan-kun Lee, Hao Zhang, Wei Chen. Capacity of soybean carbohydrate metabolism in Leuconostoc mesenteroides, Lactococcus lactis and Streptococcus thermophilus. Food Bioscience, Volume 44, Part A,
2021, 101381. ISSN 2212-4292. https://doi.org/10.1016/j.fbio.2021.101381.

7. National Food Institute, Technical University of Denmark. Frida Food Data, version 5.4, May 2026. https://frida.fooddata.dk

8. U.S. Department of Agriculture, Agricultural Research Service. FoodData Central. Version Current: May 2026. https://fdc.nal.usda.gov